# Red Nuronal para reconocer imagenes 2x2 - PyTorchOnly

Resumen/Introducción
Este trabajo presenta una implementación educativa de una red neuronal convolucional (CNN) diseñada para aprender la correspondencia entre patrones binarios de 4 bits y su representación hexadecimal (del 0x0 al 0xF). El objetivo principal es demostrar los conceptos fundamentales de las CNN, como la propagación hacia adelante (feedforward), las funciones de activación (ReLU, Softmax), la pérdida (cross-entropy) y el aprendizaje mediante retropropagación (backpropagation), utilizando únicamente Python y PyTorch para mantener un enfoque claro y minimalista.

Características del Modelo
Dataset: Todas las combinaciones posibles de 4 bits (16 muestras), tratadas como "imágenes" 1D.

Arquitectura:

Capa de entrada: 4 neuronas (una por cada bit).

Capa oculta: 8 neuronas con activación ReLU (para introducir no linealidad).

Capa de salida: 16 neuronas con activación Softmax (clasificación en 16 clases hexadecimales).

Función de pérdida: Entropía cruzada (Cross-Entropy Loss).
>La función de pérdida de entropía cruzada (Cross-Entropy Loss) es una herramienta fundamental en el aprendizaje automático, especialmente en problemas de clasificación. Mide la diferencia entre la distribución de probabilidad predicha por un modelo y la distribución de probabilidad real (ground truth). Un valor de entropía cruzada más bajo indica un mejor rendimiento del modelo, acercándose a 0 cuando el modelo predice correctament

Optimización: Descenso de gradiente (Gradient Descent).

In [23]:
# Importamos los módulos necesarios de PyTorch
import torch          # Biblioteca principal para tensores y redes neuronales
import torch.nn as nn # Contiene las clases para construir redes neuronales
import torch.optim as optim # Implementa algoritmos de optimización como SGD

# Configuramos una semilla para reproducibilidad
torch.manual_seed(42) # Esto asegura que los resultados sean los mismos en cada ejecución




### DATASET:
Vamos a crear un dataset que represente números hexadecimales (0-F) en binario.
Cada número hexadecimal se representa con 4 bits (nibble), por ejemplo:
0 -> 0000
1 -> 0001
...
F -> 1111


In [24]:

# Tensor de entrada (16 muestras × 4 características)
X = torch.tensor([
    [0, 0, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0], [0, 0, 1, 1],
    [0, 1, 0, 0], [0, 1, 0, 1], [0, 1, 1, 0], [0, 1, 1, 1],
    [1, 0, 0, 0], [1, 0, 0, 1], [1, 0, 1, 0], [1, 0, 1, 1],
    [1, 1, 0, 0], [1, 1, 0, 1], [1, 1, 1, 0], [1, 1, 1, 1]
], dtype=torch.float32) # Usamos float32 que es el tipo estándar en PyTorch

# Tensor de salida (one-hot encoding para 16 clases)
y = torch.eye(16)  # torch.eye crea una matriz identidad, perfecta para one-hot encoding


In [25]:
print(y)

tensor([[1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0


### DEFINICIÓN DEL MODELO:
Vamos a crear una red neuronal con:
- Capa de entrada: 4 neuronas (1 por cada bit)
- Capa oculta: 8 neuronas con activación ReLU
- Capa de salida: 16 neuronas (1 por cada clase hexadecimal)


In [26]:

class HexClassifier(nn.Module):
    def __init__(self):
        super().__init__() # Inicializa la clase padre nn.Module
        
        # Capa completamente conectada (linear) de 4 a 8 neuronas
        self.fc1 = nn.Linear(in_features=4, out_features=8)
        
        # Capa completamente conectada (linear) de 8 a 16 neuronas
        self.fc2 = nn.Linear(in_features=8, out_features=16)
        
        # Función de activación ReLU
        self.relu = nn.ReLU()
    
    def forward(self, x):
        # Paso 1: Propagación a través de la primera capa + activación ReLU
        x = self.fc1(x)     # Multiplicación por pesos y suma de biases
        x = self.relu(x)    # Aplicamos función de activación
        
        # Paso 2: Propagación a través de la capa de salida
        x = self.fc2(x)     # No aplicamos softmax aquí (lo incluye CrossEntropyLoss)
        
        return x



### CONFIGURACIÓN DEL ENTRENAMIENTO:
1. Creamos una instancia del modelo
2. Definimos la función de pérdida (CrossEntropyLoss)
3. Elegimos un optimizador (Stochastic Gradient Descent)


In [27]:
model = HexClassifier() # Instanciamos nuestro modelo

# CrossEntropyLoss es ideal para problemas de clasificación multiclase
# Nota: Esta función ya incluye Softmax internamente, por eso no lo aplicamos en el modelo
criterion = nn.CrossEntropyLoss()

# Optimizador SGD (Descenso de Gradiente Estocástico) con learning rate de 0.1
optimizer = optim.SGD(model.parameters(), lr=0.1)

### CICLO DE ENTRENAMIENTO:
Vamos a entrenar el modelo por 1000 épocas, mostrando el progreso cada 100 épocas

In [28]:
for epoch in range(101):
    # Paso Forward: calculamos las predicciones del modelo
    outputs = model(X)
    
    # Calculamos la pérdida comparando las predicciones con las etiquetas reales
    # y.argmax(dim=1) convierte el one-hot encoding a índices de clase (0-15)
    loss = criterion(outputs, y.argmax(dim=1))
    
    # Paso Backward: calculamos gradientes
    optimizer.zero_grad() # Limpiamos gradientes de iteraciones anteriores
    loss.backward()       # Backpropagation (calcula gradientes)
    optimizer.step()      # Actualizamos pesos según los gradientes
    
    # Mostramos progreso cada 100 épocas
    if epoch % 50 == 0:
        print(f"Época {epoch:3d}, Pérdida: {loss.item():.4f}")

Época   0, Pérdida: 2.7985


Época  50, Pérdida: 2.5921
Época 100, Pérdida: 2.3131


### EVALUACIÓN DEL MODELO:
Después del entrenamiento, evaluamos el modelo con los mismos datos
(debería aprender perfectamente este dataset pequeño)

In [29]:
hex_digits = "0123456789ABCDEF" # Dígitos hexadecimales para mostrar resultados

# Desactivamos cálculo de gradientes para evaluación (ahorra memoria y computación)
with torch.no_grad():
    # Obtenemos predicciones del modelo
    predictions = model(X)
    
    # Convertimos las predicciones a índices de clase (0-15)
    predicted_classes = predictions.argmax(dim=1)
    
    # Mostramos resultados para cada entrada
    for i in range(len(X)):
        input_data = X[i].tolist()  # Convertimos tensor a lista para mostrar
        predicted = hex_digits[predicted_classes[i]]  # Dígito hexadecimal predicho
        expected = hex_digits[i]    # Dígito hexadecimal real
        
        # Formateamos la salida para mejor visualización
        print(f"Entrada: {input_data} -> Predicho: {predicted}, Esperado: {expected}")
        

Entrada: [0.0, 0.0, 0.0, 0.0] -> Predicho: 4, Esperado: 0
Entrada: [0.0, 0.0, 0.0, 1.0] -> Predicho: 1, Esperado: 1
Entrada: [0.0, 0.0, 1.0, 0.0] -> Predicho: 3, Esperado: 2
Entrada: [0.0, 0.0, 1.0, 1.0] -> Predicho: B, Esperado: 3
Entrada: [0.0, 1.0, 0.0, 0.0] -> Predicho: D, Esperado: 4
Entrada: [0.0, 1.0, 0.0, 1.0] -> Predicho: 1, Esperado: 5
Entrada: [0.0, 1.0, 1.0, 0.0] -> Predicho: 3, Esperado: 6
Entrada: [0.0, 1.0, 1.0, 1.0] -> Predicho: 7, Esperado: 7
Entrada: [1.0, 0.0, 0.0, 0.0] -> Predicho: D, Esperado: 8
Entrada: [1.0, 0.0, 0.0, 1.0] -> Predicho: D, Esperado: 9
Entrada: [1.0, 0.0, 1.0, 0.0] -> Predicho: B, Esperado: A
Entrada: [1.0, 0.0, 1.0, 1.0] -> Predicho: B, Esperado: B
Entrada: [1.0, 1.0, 0.0, 0.0] -> Predicho: D, Esperado: C
Entrada: [1.0, 1.0, 0.0, 1.0] -> Predicho: D, Esperado: D
Entrada: [1.0, 1.0, 1.0, 0.0] -> Predicho: E, Esperado: E
Entrada: [1.0, 1.0, 1.0, 1.0] -> Predicho: F, Esperado: F



### NOTAS ADICIONALES:
1. Este es un ejemplo educativo con un dataset muy pequeño donde esperamos
   precisión del 100% ya que el problema es linealmente separable.
   
2. En problemas reales:
   - Necesitaríamos un dataset más grande
   - Dividiríamos los datos en entrenamiento y prueba
   - Usaríamos mini-batches
   - Consideraríamos regularización
   
3. PyTorch automáticamente:
   - Calcula los gradientes
   - Actualiza los pesos
   - Maneja la propagación hacia adelante y atrás


## Explicación Ampliada:
1. **Dataset:**
    - Creamos todas las combinaciones posibles de 4 bits (16 valores)
    - Las etiquetas son one-hot encoding para las 16 clases (0-F)
2. **Arquitectura del Modelo:**
    - Capa de entrada: 4 neuronas (1 por cada bit)
    - Capa oculta: 8 neuronas con activación ReLU
    - ReLU ayuda a introducir no-linealidad
    - Capa de salida: 16 neuronas (sin activación, CrossEntropyLoss aplica softmax)
3. **Entrenamiento:**
    - Forward pass: Calcula predicciones
    - Cálculo de pérdida: CrossEntropy compara predicciones con etiquetas reales
    - Backward pass:
        - zero_grad(): Limpia gradientes anteriores
        - backward(): Calcula nuevos gradientes
        - step(): Actualiza pesos
4. **Evaluación:**
    - torch.no_grad(): Desactiva cálculo de gradientes para ahorrar recursos
    - argmax(): Convierte salidas de la red en índices de clase (0-15)
5. **Visualización:**
    - Mostramos cada entrada con su predicción y valor esperado
    - Usamos una cadena "0123456789ABCDEF" para traducir índices a dígitos hexadecimales

Este código es completo pero mantiene la simplicidad para fines educativos, mostrando claramente cada paso del proceso de entrenamiento de una red neuronal.
